# Day 10 Lab: Qwen3-VL-4B QLoRA for road damage localization

Run locally using the project `.venv` Python kernel. `run_local.py` provides the same E0-E4 workflow without Jupyter. The implementation is in `student_lab.py`. Measured values in `REPORT.md` must come from actual runs.


## 0. Local environment

Create `.venv` and install `requirements-colab.txt` plus a compatible CUDA PyTorch build before opening this notebook with the `.venv` kernel. Check that CUDA is available below.


In [ ]:
from pathlib import Path
import os
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("HF_HOME", str(Path("work/hf_cache").resolve()))
import sys, torch
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# The ignored .env file can contain Kaggle credentials. Never print them.
from run_local import load_env_file
from pathlib import Path
load_env_file(Path(".env"))
DATA_ZIP = None  # Set a local dataset ZIP path if available.
DATA_SOURCE = None  # Or set a directory containing images/ and labels-YOLO/.


## 1. Dữ liệu: tải và chia split

In [ ]:
from run_local import prepare
prepare(source=DATA_SOURCE, archive=DATA_ZIP)


In [ ]:
from student_lab import inspect_splits, draw_examples
records = inspect_splits()  # asserts sessions do not cross splits
draw_examples(records['train'], count=6)

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from student_lab import yolo_to_box, objects_to_answer, image_for
row = records['train'][0]
print(objects_to_answer(row))
# Compare the green GT boxes above with the converted coordinates in pixels.
print([yolo_to_box(o['label'], o['yolo'], row['width'], row['height'], 'pixel') for o in row['objects']])

## 3. E0: zero-shot

In [ ]:
from student_lab import run_e0
result_e0 = run_e0(records['test'])
print('E0 parse_valid_rate:', result_e0['parse_valid_rate'])
from pathlib import Path
import json
with Path('work/e0.jsonl').open(encoding='utf-8') as f:
    for _, line in zip(range(5), f):
        print(json.loads(line))

In [ ]:
# The run_e0 call above has already run the supplied scorer and written evidence.
print('E0 F1:', result_e0['micro']['f1'], 'CI95:', result_e0['f1_ci95'])

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from transformers import AutoProcessor
from student_lab import MODEL_ID, encode_example, collate
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = 'right'
example = encode_example(processor, records['train'][0])
visible = example['labels'][example['labels'] != -100]
print('Assistant answer tokens:', processor.tokenizer.decode(visible, skip_special_tokens=False))
image_token_id = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
print('Image tokens:', int((example['input_ids'] == image_token_id).sum()))
print('Collated keys:', collate([example], processor.tokenizer.pad_token_id).keys())

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# E1: LLM LoRA only. E2: LLM LoRA + full projector. E3: E2 + ViT LoRA.
# apply_setup checks the expected 252/348 matched modules and prints trainable parameters.
from student_lab import apply_setup, save_checkpoint, restore_checkpoint

## 6. Vòng huấn luyện

In [ ]:
from student_lab import run_trained
results = {'E0': result_e0}
for setup in ('E1', 'E2', 'E3'):
    results[setup] = run_trained(setup, records['train'], records['test'])
    print(setup, 'F1', results[setup]['micro']['f1'],
          'CI95', results[setup]['f1_ci95'], 'meta', results[setup]['meta'])

## 7. Suy luận và chấm E1–E3

In [ ]:
# Every run_trained call reloads the saved LoRA adapter and projector (if applicable),
# predicts every test image, then calls the unmodified scripts/score.py exactly once.
for name, result in results.items():
    print(name, 'missing:', result['missing_predictions'],
          'valid:', round(result['parse_valid_rate'], 3),
          'F1:', round(result['micro']['f1'], 3),
          'CI95:', result['f1_ci95'],
          'FP/image:', round(result['fp_per_image'], 3),
          'per-class:', {k: round(v['f1'], 3) for k, v in result['per_class'].items()})

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4: rank ablation (one variable). Hold setup, data, seed, steps and prompt fixed.
# The base setup is the best observed E1-E3; both rank values are trained independently.
from student_lab import run_trained
base_setup = max(('E1', 'E2', 'E3'), key=lambda k: results[k]['micro']['f1'])
print('E4 base setup:', base_setup)
for rank in (4, 64):
    name = f'E4_rank_{rank}'
    results[name] = run_trained(name, records['train'], records['test'], setup=base_setup, rank=rank)
    print(name, 'F1', results[name]['micro']['f1'], 'CI95', results[name]['f1_ci95'])
# Fill REPORT.md from the actual report/evidence/*.json outputs and add six annotated images.
# E5 GRPO is optional bonus and is not represented as completed without a real run.

## 9. Figures and report

After test predictions exist, create six figures with green ground truth and red predictions. Fill the measured values in `REPORT.md` from `report/evidence/*.json`.


In [ ]:
from student_lab import save_illustrations
figures = save_illustrations(records["test"], "work/e3.jsonl", "e3", count=6)
print("Figures:", [str(p) for p in figures])
for name, result in results.items():
    print(name, result["meta"], result["micro"], result["f1_ci95"], result["per_class"])
